### Notebook to develop and test the implementation of MTEB for sparse vectorization

In [1]:
import re 
import mteb
from src import tfidf_for_mteb
#from threadpoolctl import threadpool_limits

# autoreload to keep track of changes
#%load_ext autoreload
#%autoreload 1
#%aimport src.tfidf_for_mteb 

#### Step 1: write a function to get a vocabulary list from a list of strings

In [ ]:
# using regex to find words for the vocabulary list 
# This function needs to be callable by the evaluators in sparse_mteb/mteb/evaluation
# therefore, it is copied into sparse_mteb/mteb/evaluation/evaluators/utils.py

def get_vocab(text: [str], token_pattern: str = r"(?u)\b\w\w+\b", lowercase: bool = True) -> [str]:
    """return a list of unique words ocurring in text that fulfill the specified token_pattern
    The default token_pattern is the one used in the scikit-learn TfidfVectorizer class
    """
    if lowercase:
        vocab = [word for sent in text for word in re.findall(token_pattern, sent.lower())]
    else:
        vocab = [word for sent in text for word in re.findall(token_pattern, sent)]
    
    return list(set(vocab))


In [ ]:
test_list = ["These are some words. What else?", "I've thought hard about what words to write.", "Words won't be enough!"]
get_vocab(text=test_list)

#### Step 2: Try mteb tasks with tfidf

In [2]:
# define models to compare
tfidf_model = tfidf_for_mteb.Tfidf()
glove_model = mteb.get_model("sentence-transformers/average_word_embeddings_glove.6B.300d")

##### Bitext Mining 

Task: translation: match sentence from set 1 to sentence from set 2 (in different language)

Implementation for TF-IDF: tfidf is obviously horrible at this. the sets could either be embedded together (which is how I implemented it for now), which means that the translation for each word would just be another matrix entry, or embedded separately, which also makes no sense because the first word in the english vocab will not align with the first word in french 

Changes implemented in which files?

In [ ]:
tasks = mteb.get_tasks(tasks=["FloresBitextMining"])
evaluation = mteb.MTEB(tasks=tasks)
tfidf_results = evaluation.run(tfidf_model, output_folder="../MTEB/sparse_results")


In [ ]:
glove_results = evaluation.run(glove_model, output_folder="../MTEB/sparse_results")

##### Classification

**Task**: train logistic regression classifier on train set embeddings, evaluate on performance for test set embeddings

**Implementation for TF-IDF:** compute vocab for embeddings for the union of train and test set. For multilingual datasets, the embeddings are computed for each language separately to reduce the size of embeddings (this shouldn't lead to conflicts because each language is evaluated separately)

**Changes implemented in which files?**
AbsTaskClassification.py

In [ ]:
tasks = mteb.get_tasks(tasks=["Banking77Classification"])
evaluation = mteb.MTEB(tasks=tasks)

In [ ]:
tfidf_results = evaluation.run(tfidf_model, output_folder="../MTEB/sparse_results")

In [ ]:
glove_results = evaluation.run(glove_model, output_folder="../MTEB/sparse_results")

##### Clustering

**Task**: k-means model on labeled embeddings for k clusters

**Implementation for TF-IDF**: MTEB only performs clustering on small subset of data (2048 samples) -> should we compute embeddings for this subset or entire data? (This is at least the case for the tasks that get evaluated with AbsTaskClusteringFast.py, maybe not those that get evaluated with AbsTaskClustering.py, but how do I differentiate between these?)

Embedding space is created for the entire corpus, not only the current subset. This is conceptually stronger, because then all clustering happens on the same subspace. Performance-wise, this doesn't change much, speed-wise it does.


**Changes implemented in which files?**
(All of these changes are currently commented out)
- AbsTask.py (this will most likely lead to conflicts!)
- AbsTaskClusteringFast.py
- AbsTaskClustering.py

In [ ]:
tasks = mteb.get_tasks(tasks=["MedrxivClusteringP2P"])
evaluation = mteb.MTEB(tasks=tasks)

TF-IDF on ArXivHierarchicalClusteringS2S

results for computing embeddings on subset: v-measure: 0.4975645682072562

results for computing embeddings on whole data: "v_measure": 0.5006167526797055

TF-IDF on ArxivClusteringS2S

results for computing embeddings on whole data: "v_measure": 0.11543345526606984

TF-IDF on MedrivClusteringS2S

results for computing embeddings on subset: "v_measure": 0.15541653902737848

results for computing embeddings on whole data: 0.15547118002961202

In [ ]:
tfidf_results = evaluation.run(tfidf_model, output_folder="../MTEB/sparse_results")

##### Pair Classification

**Task**: classify a pair of texts as duplicates or paraphrases (binary choice)

**Implementation for TF-IDF**: probably bad performance because tfidf would only recognize a parashrase pair if the same words are used in both. Embedding should be done on the whole dataset, not on the single pairs.

**Changes implemented in which files?**
No changes necessary, embedding already happens for the entire dataset at once

In [ ]:
tasks = mteb.get_tasks(tasks=["TwitterSemEval2015"])
evaluation = mteb.MTEB(tasks=tasks)

In [ ]:
tfidf_results = evaluation.run(tfidf_model, output_folder="../MTEB/sparse_results")

##### Reranking

**Task**: rank texts according to relevance to query. relevance is computed for each document-query pair. each query has a list of positive/relevant and negative/irrelevant documents

**Implementation for TF-IDF**: embed all texts & all queries together

**Changes implemented in which files?**
- RerankingEvaluator.py

In [ ]:
tasks = mteb.get_tasks(tasks=["AskUbuntuDupQuestions"])
evaluation = mteb.MTEB(tasks=tasks)

In [ ]:
tfidf_results = evaluation.run(tfidf_model, output_folder="../MTEB/sparse_results")

##### Retrieval

**Task**: select texts that are relevant for query

**Implementation for TF-IDF**: embed all texts & all queries together

**Changes implemented in which files?**
- AbsTaskRetrieval.py


In [ ]:
tasks = mteb.get_tasks(tasks=["NFCorpus"])
evaluation = mteb.MTEB(tasks=tasks)

In [ ]:
tfidf_results = evaluation.run(tfidf_model, output_folder="../MTEB/sparse_results")

##### STS (Semantic Textual Similarity)

**Task**: determine similarity between sentence pairs, with ground truth similarities given

**Implementation for TF-IDF**: embed all sentence pairs together or each pair on its own? Maybe test both?

**Changes implemented in which files?**
- STSEvaluator.py

In [ ]:
tasks = mteb.get_tasks(tasks=["BIOSSES"])
evaluation = mteb.MTEB(tasks=tasks)

In [ ]:
tfidf_results = evaluation.run(tfidf_model, output_folder="../MTEB/sparse_results")

##### Summarization

**Task**: Machine-generated summaries are provided with human-generated quality scores. Task is to predict the summary quality by embedding the machine-generated summaries and corresponding human-generated summaries, and for each summary use the minimal distance to one of the human-generated summaries as a score.

**Implementation for TF-IDF**: 

**Changes implemented in which files?**


In [5]:
tasks = mteb.get_tasks(tasks=["SummEvalSummarization.v2"])
evaluation = mteb.MTEB(tasks=tasks)

In [6]:
tfidf_results = evaluation.run(tfidf_model, output_folder="../MTEB/sparse_results")

───────────────────────────────────────────────── Selected tasks  ─────────────────────────────────────────────────

Summarization

- SummEvalSummarization.v2, p2p

AbsTask.load_data called


Repo card metadata block was not found. Setting CardData to empty.


dataset before transform: DatasetDict({
    test: Dataset({
        features: ['machine_summaries', 'human_summaries', 'relevance', 'coherence', 'fluency', 'consistency', 'text', 'id'],
        num_rows: 100
    })
})
dataset keys: dict_keys(['test'])
dataset['test']: Dataset({
    features: ['machine_summaries', 'human_summaries', 'relevance', 'coherence', 'fluency', 'consistency', 'text', 'id'],
    num_rows: 100
})
self.metadata_dict: {'dataset': {'path': 'mteb/summeval', 'revision': 'cda12ad7615edc362dbf25a00fdd61d3b1eaf93c'}, 'name': 'SummEvalSummarization.v2', 'description': 'News Article Summary Semantic Similarity Estimation. This version fixes a bug in the evaluation script that caused the main score to be computed incorrectly.', 'type': 'Summarization', 'modalities': ['text'], 'category': 'p2p', 'reference': 'https://github.com/Yale-LILY/SummEval', 'eval_splits': ['test'], 'eval_langs': ['eng-Latn'], 'main_score': 'cosine_spearman', 'date': ('2016-01-01', '2016-12-31'), 'doma

Scoring: 100%|██████████| 100/100 [00:01<00:00, 54.91it/s]
